# **Notebook 1** -- Data Acquisition

En este notebook se obtendran los datos historicos de todos los activos seleccionados desde Yahoo Finance, realizar una validacion inicial de la informacion descargada y almacenar los datos en un formato eficiente para su posterior analisis

### **Objetivo de negocio**

Obtener información histórica confiable de los activos financieros seleccionados para construir una base de datos consistente que permita realizar análisis cuantitativos y diseñar un portafolio de inversión acorde con el perfil del cliente.

### **Objetivo analitico**

Descargar los precios históricos de los activos que conforman el universo de inversión desde Yahoo Finance, validar la integridad y consistencia de los datos obtenidos y almacenarlos en un formato eficiente para su posterior análisis.

### **Preguntas a responder en este notebook**

- ¿Todos los tickers existen?
- ¿Todos descargaron datos?
- ¿Todos tienen el mismo rango de tiempo?
- ¿Hay valores o datos que faltan?
- ¿Que columnas entrega Yahoo Finance?

### **Entregable de este notebook**

Este notebook no entrega gráficos ni conclusiones de inversión.

Su producto final será:

- El archivo prices.parquet almacenado en data/raw/.
- La validación inicial de la calidad de los datos.
- La confirmación de que el universo de inversión puede analizarse correctamente.

Ese será el insumo para el Notebook 02.

### **Importacion de librerias**

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import pyarrow as pa

In [18]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [19]:
# Project Configuration

START_DATE = "2021-06-01"
END_DATE = "2026-06-01"

BENCHMARK = "^GSPC"

RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed/"

Se seleccionó el período comprendido entre junio de 2021 y junio de 2026 para utilizar los cinco años más recientes de información disponible. Este horizonte proporciona datos representativos del comportamiento reciente del mercado, incluyendo diferentes condiciones económicas, y evita depender de información demasiado antigua que podría no reflejar el entorno actual de inversión.

In [20]:
# Load Investment Universe

investment_universe = pd.read_csv(
    RAW_DATA_PATH + "investment_universe.csv"
)

investment_universe.head()

,ticker,asset_name,asset_class,sector
0,NVDA,Nvidia,Equity - Large Cap,Semiconductors
1,AAPL,Apple,Equity - Large Cap,Technology Hardware
2,GOOGL,Alphabet (Google),Equity - Large Cap,Internet & Media
3,AVGO,Broadcom,Equity - Large Cap,Semiconductors
4,BRK-B,Berkshire Hathaway,Equity - Large Cap,Diversified Holdings


In [21]:
# Initial Validation

print(f"Numero de activos: {investment_universe.shape[0]}")
print(f"Numero de columnas: {investment_universe.shape[1]}")

investment_universe.info()

Numero de activos: 32
Numero de columnas: 4
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ticker       32 non-null     str  
 1   asset_name   32 non-null     str  
 2   asset_class  32 non-null     str  
 3   sector       32 non-null     str  
dtypes: str(4)
memory usage: 2.5 KB


In [22]:
investment_universe.sample(5)

,ticker,asset_name,asset_class,sector
19,COUR,Coursera,Equity - Growth,EdTech
14,GXO,GXO Logistics,Equity - Growth,Logistics
15,W,Wayfair,Equity - Growth,E-commerce
6,LLY,Eli Lilly,Equity - Large Cap,Pharmaceuticals
27,GC=F,Oro,Commodity,Metales Preciosos


In [23]:
# Extract Tickers

tickers = investment_universe["ticker"].tolist()

print(tickers)

['NVDA', 'AAPL', 'GOOGL', 'AVGO', 'BRK-B', 'MU', 'LLY', 'GE', 'NEE', 'GEV', 'BEP', 'UPS', 'FDX', 'JBHT', 'GXO', 'W', 'CHWY', 'RVLV', 'ETSY', 'COUR', 'DUOL', 'PSO', 'JPM', 'KO', 'MS', 'PFE', 'DE', 'GC=F', 'CL=F', 'KC=F', 'CC=F', 'IEF']


In [24]:
# Import function to download data from Yahoo Finance

from src.data_loader import download_market_data

In [25]:
# Download Historical Market Data

prices = download_market_data(
    tickers=tickers,
    start_date=START_DATE,
    end_date=END_DATE,
)

In [26]:
prices.head()

Price            Close                                                       \
Ticker            AAPL       AVGO        BEP       BRK-B    CC=F       CHWY   
Date                                                                          
2021-06-01  121.142174  42.303005  31.872864  289.839996  2431.0  73.529999   
2021-06-02  121.902466  42.792599  32.009956  290.019989  2426.0  77.519997   
2021-06-03  120.420853  41.985935  31.453472  291.970001  2415.0  75.129997   
2021-06-04  122.711510  42.907314  31.598648  292.519989  2409.0  75.209999   
2021-06-07  122.721268  41.893799  31.276051  289.459991  2350.0  78.559998   

Price                                              ...      Volume             \
Ticker           CL=F       COUR          DE DUOL  ...         LLY         MS   
Date                                               ...                          
2021-06-01  67.720001  38.529999  340.536133  NaN  ...   3070200.0  9576400.0   
2021-06-02  68.830002  39.959999  333.157623  NaN  ...   2030300.0  6878400.0   
2021-06-03  68.809998  39.599998  335.212463  NaN  ...   3066600.0  7926800.0   
2021-06-04  69.620003  39.650002  333.092346  NaN  ...   2817700.0  6406800.0   
2021-06-07  69.230003  41.669998  331.962250  NaN  ...  17231400.0  6773800.0   

Price                                                                 \
Ticker              MU        NEE         NVDA         PFE       PSO   
Date                                                                   
2021-06-01  11067200.0  9056100.0  472804000.0  23642400.0  186500.0   
2021-06-02  10314200.0  6421000.0  594168000.0  19629000.0  203700.0   
2021-06-03  14742100.0  8871500.0  580008000.0  17376300.0  298400.0   
2021-06-04  12591100.0  6586300.0  617120000.0  19375300.0  139800.0   
2021-06-07   9006400.0  5988500.0  575756000.0  24110700.0  108600.0   

Price                                        
Ticker           RVLV        UPS          W  
Date                                         
2021-06-01   887600.0  1865300.0  1332000.0  
2021-06-02  1060000.0  2486000.0  1615800.0  
2021-06-03  1212100.0  2172400.0  1244100.0  
2021-06-04   539600.0  2943900.0   897200.0  
2021-06-07  1334500.0  3444300.0  1752000.0  

[5 rows x 160 columns]

Durante la adquisición de datos se identificó que UDMY dejó de cotizar durante el período del proyecto debido a una adquisición corporativa. En consecuencia, se excluye del universo de inversión, ya que no dispone de una serie histórica continua para el horizonte de análisis.

### **Data quality assessment**

In [27]:
# Close Prices

close_prices = prices["Close"].copy()

In [28]:
# Missing Values (%)

missing_percentage = (
    close_prices.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_percentage

Ticker
GEV      56.677266
DUOL      3.418124
GXO       3.100159
AAPL      0.238474
CHWY      0.238474
COUR      0.238474
BEP       0.238474
AVGO      0.238474
FDX       0.238474
GE        0.238474
ETSY      0.238474
DE        0.238474
GOOGL     0.238474
JBHT      0.238474
IEF       0.238474
BRK-B     0.238474
PSO       0.238474
NEE       0.238474
NVDA      0.238474
JPM       0.238474
KO        0.238474
LLY       0.238474
MS        0.238474
MU        0.238474
UPS       0.238474
RVLV      0.238474
W         0.238474
PFE       0.238474
GC=F      0.079491
CL=F      0.079491
CC=F      0.079491
KC=F      0.000000
dtype: float64

In [29]:
observations = close_prices.count().sort_values()

observations

Ticker
GEV       545
DUOL     1215
GXO      1219
AAPL     1255
CHWY     1255
COUR     1255
BEP      1255
AVGO     1255
FDX      1255
GE       1255
ETSY     1255
DE       1255
GOOGL    1255
JBHT     1255
IEF      1255
BRK-B    1255
PSO      1255
NEE      1255
NVDA     1255
JPM      1255
KO       1255
LLY      1255
MS       1255
MU       1255
UPS      1255
RVLV     1255
W        1255
PFE      1255
GC=F     1257
CL=F     1257
CC=F     1257
KC=F     1258
dtype: int64

In [30]:
missing_percentage[missing_percentage > 0]

Ticker
GEV      56.677266
DUOL      3.418124
GXO       3.100159
AAPL      0.238474
CHWY      0.238474
COUR      0.238474
BEP       0.238474
AVGO      0.238474
FDX       0.238474
GE        0.238474
ETSY      0.238474
DE        0.238474
GOOGL     0.238474
JBHT      0.238474
IEF       0.238474
BRK-B     0.238474
PSO       0.238474
NEE       0.238474
NVDA      0.238474
JPM       0.238474
KO        0.238474
LLY       0.238474
MS        0.238474
MU        0.238474
UPS       0.238474
RVLV      0.238474
W         0.238474
PFE       0.238474
GC=F      0.079491
CL=F      0.079491
CC=F      0.079491
dtype: float64

La mayoría de los activos presentan una cobertura prácticamente completa durante el período analizado. Los valores faltantes observados se clasifican en dos grupos:

- Faltantes estructurales, asociados a empresas que comenzaron a cotizar después del inicio del período de análisis (por ejemplo, GE Vernova, Duolingo y GXO Logistics).
- Faltantes puntuales, inferiores al 0.3%, atribuibles a diferencias en calendarios de negociación entre mercados o a días sin negociación. Debido a su baja magnitud, no representan un problema significativo para el análisis.

In [31]:
first_valid_date = close_prices.apply(lambda column: column.first_valid_index())

first_valid_date

Ticker
AAPL    2021-06-01
AVGO    2021-06-01
BEP     2021-06-01
BRK-B   2021-06-01
CC=F    2021-06-01
CHWY    2021-06-01
CL=F    2021-06-01
COUR    2021-06-01
DE      2021-06-01
DUOL    2021-07-28
ETSY    2021-06-01
FDX     2021-06-01
GC=F    2021-06-01
GE      2021-06-01
GEV     2024-03-27
GOOGL   2021-06-01
GXO     2021-07-22
IEF     2021-06-01
JBHT    2021-06-01
JPM     2021-06-01
KC=F    2021-06-01
KO      2021-06-01
LLY     2021-06-01
MS      2021-06-01
MU      2021-06-01
NEE     2021-06-01
NVDA    2021-06-01
PFE     2021-06-01
PSO     2021-06-01
RVLV    2021-06-01
UPS     2021-06-01
W       2021-06-01
dtype: datetime64[s]

In [32]:
last_valid_date = close_prices.apply(lambda column: column.last_valid_index())

last_valid_date

Ticker
AAPL    2026-05-29
AVGO    2026-05-29
BEP     2026-05-29
BRK-B   2026-05-29
CC=F    2026-05-29
CHWY    2026-05-29
CL=F    2026-05-29
COUR    2026-05-29
DE      2026-05-29
DUOL    2026-05-29
ETSY    2026-05-29
FDX     2026-05-29
GC=F    2026-05-29
GE      2026-05-29
GEV     2026-05-29
GOOGL   2026-05-29
GXO     2026-05-29
IEF     2026-05-29
JBHT    2026-05-29
JPM     2026-05-29
KC=F    2026-05-29
KO      2026-05-29
LLY     2026-05-29
MS      2026-05-29
MU      2026-05-29
NEE     2026-05-29
NVDA    2026-05-29
PFE     2026-05-29
PSO     2026-05-29
RVLV    2026-05-29
UPS     2026-05-29
W       2026-05-29
dtype: datetime64[s]

In [33]:
# Data Quality Summary

data_quality = pd.DataFrame({
    "first_date": first_valid_date,
    "last_date": last_valid_date,
    "observations": close_prices.count(),
    "missing_percentage": missing_percentage
})

data_quality.sort_values("missing_percentage", ascending=False)

,first_date,last_date,observations,missing_percentage
Ticker,,,,
GEV,2024-03-27,2026-05-29,545,56.677266
DUOL,2021-07-28,2026-05-29,1215,3.418124
GXO,2021-07-22,2026-05-29,1219,3.100159
AAPL,2021-06-01,2026-05-29,1255,0.238474
CHWY,2021-06-01,2026-05-29,1255,0.238474
COUR,2021-06-01,2026-05-29,1255,0.238474
BEP,2021-06-01,2026-05-29,1255,0.238474
AVGO,2021-06-01,2026-05-29,1255,0.238474
FDX,2021-06-01,2026-05-29,1255,0.238474


### **Resumen de la adquisición de datos**
**Objetivo**

Descargar y validar datos históricos de mercado para el universo de inversión seleccionado.

**Hallazgos**
- Se descargaron con éxito los datos históricos de todos los activos seleccionados, a excepción de UDMY, que fue excluido del universo de inversión tras ser adquirido y retirado de cotización.
- La mayoría de los activos presentan una cobertura histórica completa, con un porcentaje de valores faltantes inferior al 0,3 %.
- DUOL y GXO presentan algunos valores faltantes debido a que comenzaron a cotizar después del inicio del periodo de análisis seleccionado.
- GEV muestra un registro histórico considerablemente más breve debido a su reciente salida a bolsa, lo que resulta en aproximadamente un 56,7 % de observaciones faltantes.
- Más adelante en el proyecto se realizará un análisis de sensibilidad que incluya a GEV para evaluar su impacto en la optimización de la cartera.

**Siguiente paso**

El conjunto de datos validado se utilizará en el siguiente *notebook* para evaluar el universo de inversión y definir el conjunto final de activos aptos para la optimización de la cartera.

In [39]:
import importlib
import src.data_loader

importlib.reload(src.data_loader)

<module 'src.data_loader' from 'c:\\Users\\nicol\\OneDrive\\Desktop\\portfolio-optimization\\src\\data_loader.py'>

In [40]:
from src.data_loader import save_dataset

In [41]:
# Save Datasets

save_dataset(
    prices,
    PROCESSED_DATA_PATH + "market_data_raw.parquet",
)

save_dataset(
    close_prices,
    PROCESSED_DATA_PATH + "close_prices.parquet",
)

In [42]:
# Verification

verification = pd.read_parquet(
    PROCESSED_DATA_PATH + "close_prices.parquet"
)

verification.head()

Ticker,AAPL,AVGO,BEP,BRK-B,CC=F,CHWY,CL=F,COUR,DE,DUOL,...,LLY,MS,MU,NEE,NVDA,PFE,PSO,RVLV,UPS,W
Date,,,,,,,,,,,,,,,,,,,,,
2021-06-01,121.142174,42.303005,31.872864,289.839996,2431.0,73.529999,67.720001,38.529999,340.536133,NaN,...,189.276047,78.408318,82.057800,63.236561,16.204721,29.796623,10.627218,58.740002,170.162445,318.000000
2021-06-02,121.902466,42.792599,32.009956,290.019989,2426.0,77.519997,68.830002,39.959999,333.157623,NaN,...,189.953278,78.961617,82.243050,63.753868,16.716581,30.021057,10.494267,59.110001,169.061554,332.890015
2021-06-03,120.420853,41.985935,31.453472,291.970001,2415.0,75.129997,68.809998,39.599998,335.212463,NaN,...,193.358765,79.463875,79.990479,63.455750,16.907379,30.160370,10.520860,58.459999,169.117340,325.579987
2021-06-04,122.711510,42.907314,31.598648,292.519989,2409.0,75.209999,69.620003,39.650002,333.092346,NaN,...,192.710098,79.983124,81.677483,63.429432,17.513645,30.299681,10.644947,56.509998,168.040329,321.029999
2021-06-07,122.721268,41.893799,31.276051,289.459991,2350.0,78.559998,69.230003,41.669998,331.962250,NaN,...,212.265411,79.344688,81.950516,63.543430,17.554241,30.175858,10.742444,54.450001,169.867264,327.950012
